# M1 Colab CUDA FP16 smoke test

Select **Runtime → Change runtime type → L4 GPU** (or **T4 GPU** on free Colab) before running. This notebook clones the requested branch, installs CUDA-side Python dependencies (including `bitsandbytes`), runs the repository preflight, loads the pinned FP16 checkpoint, runs one synthetic image-question inference, and saves peak GPU memory measurements. It does not construct or evaluate an NF4 model.

The `Gauri-Commits` branch must be published to GitHub before Colab can clone it.

In [ ]:
import subprocess

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
)
print(gpu_check.stdout.strip())
gpu_name = gpu_check.stdout.splitlines()[0].split(",", 1)[0].strip()
SUPPORTED_GPUS = ("L4", "T4")
if not any(supported in gpu_name for supported in SUPPORTED_GPUS):
    raise RuntimeError(f"Expected a Colab L4 or T4 GPU; found {gpu_name!r}. Select a GPU runtime and rerun.")
GPU_TAG = next(supported for supported in SUPPORTED_GPUS if supported in gpu_name).lower()

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/Omkar38/mixed-precision-visual-shift.git"
BRANCH = "Gauri-Commits"
REPO_DIR = Path("/content/mixed-precision-visual-shift")
if REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} already exists. Restart the Colab runtime before cloning again.")
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Checked out {BRANCH} at {commit}")
%cd /content/mixed-precision-visual-shift

In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-enabled PyTorch build; install the remaining runtime dependencies.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "transformers==4.52.4",
        "accelerate==1.15.0",
        "bitsandbytes",
        "Pillow",
        "sentencepiece",
        "num2words",
    ],
    check=True,
)

In [ ]:
from pathlib import Path
import subprocess
import sys

Path("artifacts").mkdir(exist_ok=True)
subprocess.run(
    [sys.executable, "scripts/preflight.py", "--output", f"artifacts/preflight-colab-{GPU_TAG}.json"],
    check=True,
)

In [ ]:
import importlib.metadata
import torch
import bitsandbytes

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime and restart the session.")
if torch.cuda.get_device_name(0) != gpu_name:
    raise RuntimeError("The CUDA device does not match the GPU reported by nvidia-smi.")
package_versions = {
    name: importlib.metadata.version(name)
    for name in ("torch", "transformers", "accelerate", "bitsandbytes", "Pillow", "sentencepiece", "num2words")
}
print({"gpu": torch.cuda.get_device_name(0), "torch_cuda": torch.version.cuda, "packages": package_versions})

In [ ]:
import time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
MODEL_REVISION = "482adb537c021c86670beed01cd58990d01e72e4"
DEVICE = "cuda:0"
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats(0)
load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_started
if next(model.parameters()).dtype != torch.float16:
    raise RuntimeError(f"Expected FP16 parameters; found {next(model.parameters()).dtype}.")
load_peak_allocated_bytes = torch.cuda.max_memory_allocated(0)
load_peak_reserved_bytes = torch.cuda.max_memory_reserved(0)
print({
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "parameter_dtype": str(next(model.parameters()).dtype),
    "load_seconds": load_seconds,
    "load_peak_allocated_bytes": load_peak_allocated_bytes,
    "load_peak_reserved_bytes": load_peak_reserved_bytes,
})

In [ ]:
from PIL import Image

image = Image.new("RGB", (384, 384), color=(32, 96, 160))
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Is the image blue? Answer yes or no."},
    ],
}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {
    key: value.to(device=DEVICE, dtype=next(model.parameters()).dtype)
    if value.is_floating_point()
    else value.to(DEVICE)
    for key, value in inputs.items()
}
prompt_length = inputs["input_ids"].shape[-1]
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats(0)
inference_started = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, do_sample=False, max_new_tokens=4)
torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_started
answer = processor.batch_decode(
    generated[:, prompt_length:], skip_special_tokens=True
)[0].strip()
inference_peak_allocated_bytes = torch.cuda.max_memory_allocated(0)
inference_peak_reserved_bytes = torch.cuda.max_memory_reserved(0)
print({
    "synthetic_input": True,
    "answer": answer,
    "do_sample": False,
    "max_new_tokens": 4,
    "inference_seconds": inference_seconds,
    "inference_peak_allocated_bytes": inference_peak_allocated_bytes,
    "inference_peak_reserved_bytes": inference_peak_reserved_bytes,
})

In [ ]:
import json
from pathlib import Path

report = {
    "status": "synthetic_fp16_smoke_test_complete",
    "branch": BRANCH,
    "repository_commit": commit,
    "gpu": torch.cuda.get_device_name(0),
    "torch_cuda": torch.version.cuda,
    "packages": package_versions,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "parameter_dtype": str(next(model.parameters()).dtype),
    "load_seconds": load_seconds,
    "load_peak_allocated_bytes": load_peak_allocated_bytes,
    "load_peak_reserved_bytes": load_peak_reserved_bytes,
    "synthetic_input": True,
    "answer": answer,
    "inference_seconds": inference_seconds,
    "inference_peak_allocated_bytes": inference_peak_allocated_bytes,
    "inference_peak_reserved_bytes": inference_peak_reserved_bytes,
    "quantization_audited": False,
}
report_path = Path(f"artifacts/m1-colab-{GPU_TAG}-fp16-smoke.json")
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved measurements to {report_path}")
print(json.dumps(report, indent=2))